In [1]:
pip install feedparser requests


     ---------------------------------------- 0.0/46.7 kB ? eta -:--:--
     ---------------------------------------- 46.7/46.7 kB 1.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/80.7 kB ? eta -:--:--
   ---------------------------------------- 80.7/80.7 kB 4.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/73.1 kB ? eta -:--:--
   ---------------------------------------- 73.1/73.1 kB 3.9 MB/s eta 0:00:00
   ---------------------------------------- 0.0/137.0 kB ? eta -:--:--
   ---------------------------------------- 137.0/137.0 kB 7.9 MB/s eta 0:00:00
   ---------------------------------------- 0.0/200.6 kB ? eta -:--:--
   ---------------------------------------- 200.6/200.6 kB ? eta 0:00:00
   ---------------------------------------- 0.0/68.5 kB ? eta -:--:--
   ---------------------------------------- 68.5/68.5 kB 3.9 MB/s eta 0:00:00
   ---------------------------------------- 0.0/131.1 kB ? eta -:--:--
   -----------------------------------


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
pip freeze > requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install pandas

   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
    --------------------------------------- 0.2/9.8 MB 5.3 MB/s eta 0:00:02
   -- ------------------------------------- 0.7/9.8 MB 9.1 MB/s eta 0:00:02
   ----- ---------------------------------- 1.4/9.8 MB 11.3 MB/s eta 0:00:01
   --------- ------------------------------ 2.4/9.8 MB 13.6 MB/s eta 0:00:01
   -------------- ------------------------- 3.5/9.8 MB 16.1 MB/s eta 0:00:01
   -------------------- ------------------- 5.1/9.8 MB 19.1 MB/s eta 0:00:01
   ---------------------------- ----------- 7.0/9.8 MB 22.5 MB/s eta 0:00:01
   -------------------------------------- - 9.5/9.8 MB 26.4 MB/s eta 0:00:01
   ---------------------------------------- 9.8/9.8 MB 26.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/12.5 MB ? eta -:--:--
   --------- ------------------------------ 3.0/12.5 MB 64.6 MB/s eta 0:00:01
   ---------------------- ----------------- 6.9/12.5 MB 74.3 MB/s eta 0:00:01
   -----------


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
import requests
import feedparser
import pandas as pd

In [13]:
sources = {
    "Página 12 - Ciencia":
        "https://www.pagina12.com.ar/arc/outboundfeeds/rss/secciones/ciencia/notas",

    "Página 12 - Universidad":
        "https://www.pagina12.com.ar/arc/outboundfeeds/rss/secciones/universidad/notas",

    "Agencia UNQ - Cat 15":
        "https://agencia.unq.edu.ar/?feed=rss2&cat=15",

    "Agencia UNQ - Cat 8":
        "https://agencia.unq.edu.ar/?feed=rss2&cat=8",

    "Agencia UNQ - Cat 482":
        "https://agencia.unq.edu.ar/?feed=rss2&cat=482",

    "UTN - Revista Tecnología y Ciencia":
        "https://rtyc.utn.edu.ar/index.php/rtyc/gateway/plugin/WebFeedGatewayPlugin/rss2",
}

In [14]:
def validate_rss(name, url):
    try:
        response = requests.get(
            url,
            timeout=15,
            headers={
                "User-Agent": "Mozilla/5.0"
            }
        )

        feed = feedparser.parse(response.content)

        return {
            "source": name,
            "url": url,
            "http_status": response.status_code,
            "content_type": response.headers.get("Content-Type"),
            "feed_title": feed.feed.get("title"),
            "entries": len(feed.entries),
            "bozo": feed.bozo,
            "valid": response.ok and len(feed.entries) > 0
        }

    except Exception as error:
        return {
            "source": name,
            "url": url,
            "http_status": None,
            "content_type": None,
            "feed_title": None,
            "entries": 0,
            "bozo": None,
            "valid": False,
            "error": str(error)
        }

In [15]:
results = [
    validate_rss(name, url)
    for name, url in sources.items()
]

df_results = pd.DataFrame(results)

df_results

,source,url,http_status,content_type,feed_title,entries,bozo,valid
0,Página 12 - Ciencia,https://www.pagina12.com.ar/arc/outboundfeeds/...,200,application/xml; charset=utf-8,Ciencia | Página|12,25,False,True
1,Página 12 - Universidad,https://www.pagina12.com.ar/arc/outboundfeeds/...,200,application/xml; charset=utf-8,Universidad | Página|12,25,False,True
2,Agencia UNQ - Cat 15,https://agencia.unq.edu.ar/?feed=rss2&cat=15,200,application/rss+xml; charset=UTF-8,Tecnología - Agencia de Noticias Científicas d...,12,False,True
3,Agencia UNQ - Cat 8,https://agencia.unq.edu.ar/?feed=rss2&cat=8,200,application/rss+xml; charset=UTF-8,Ciencias - Agencia de Noticias Científicas de ...,12,False,True
4,Agencia UNQ - Cat 482,https://agencia.unq.edu.ar/?feed=rss2&cat=482,200,application/rss+xml; charset=UTF-8,Empresas - Agencia de Noticias Científicas de ...,12,False,True
5,UTN - Revista Tecnología y Ciencia,https://rtyc.utn.edu.ar/index.php/rtyc/gateway...,200,text/html; charset=utf-8,Revista Tecnología y Ciencia,15,False,True


In [16]:
def show_feed(url, limit=5):
    response = requests.get(
        url,
        timeout=15,
        headers={"User-Agent": "Mozilla/5.0"}
    )

    feed = feedparser.parse(response.content)

    data = []

    for entry in feed.entries[:limit]:
        data.append({
            "title": entry.get("title"),
            "published": entry.get("published"),
            "link": entry.get("link")
        })

    return pd.DataFrame(data)

In [17]:
show_feed(
    sources["Página 12 - Ciencia"],
    limit=10
)

,title,published,link
0,Hallazgo en La Patagonia: un cráneo de 66 mill...,"Wed, 24 Jun 2026 23:59:38 +0000",https://www.pagina12.com.ar/2026/06/24/hallazg...
1,Científicos argentinos abren la posibilidad de...,"Sun, 16 Nov 2025 19:53:11 +0000",https://www.pagina12.com.ar/2025/11/16/cientif...
2,Alarma por el retroceso del glaciar Perito Moreno,"Thu, 07 Aug 2025 23:41:22 +0000",https://www.pagina12.com.ar/848097-alarma-por-...
3,"La investigadora del CONICET Sandra Díaz, entr...","Thu, 17 Apr 2025 00:52:24 +0000",https://www.pagina12.com.ar/818840-la-investig...
4,El futuro de la IA y su impacto en el conocimi...,"Sat, 19 Oct 2024 03:01:00 +0000",https://www.pagina12.com.ar/775639-el-futuro-d...
5,La NASA confirma la existencia de túneles en l...,"Wed, 17 Jul 2024 13:51:06 +0000",https://www.pagina12.com.ar/752825-la-nasa-con...
6,"Descubren un planeta ""demasiado masivo para su...","Sun, 03 Dec 2023 19:06:18 +0000",https://www.pagina12.com.ar/691528-descubren-u...
7,"China: científicos desarrollan una ""capa de in...","Mon, 27 Nov 2023 21:38:12 +0000",https://www.pagina12.com.ar/689691-china-cient...
8,Científicos del CONICET descubrieron una nueva...,"Thu, 09 Nov 2023 20:40:54 +0000",https://www.pagina12.com.ar/614939-cientificos...
9,Japón: registran el momento en el que nace una...,"Wed, 08 Nov 2023 22:53:11 +0000",https://www.pagina12.com.ar/614543-japon-regis...


In [18]:
def explore_feeds(sources, entries_per_feed=5):
    data = []

    for source_name, url in sources.items():
        try:
            response = requests.get(
                url,
                timeout=15,
                headers={"User-Agent": "Mozilla/5.0"}
            )

            feed = feedparser.parse(response.content)

            for entry in feed.entries[:entries_per_feed]:
                data.append({
                    "source": source_name,
                    "title": entry.get("title"),
                    "published": entry.get("published"),
                    "link": entry.get("link")
                })

        except Exception as error:
            data.append({
                "source": source_name,
                "title": None,
                "published": None,
                "link": None,
                "error": str(error)
            })

    return pd.DataFrame(data)

In [19]:
df_feed_exploration = explore_feeds(
    sources,
    entries_per_feed=5
)

df_feed_exploration

,source,title,published,link
0,Página 12 - Ciencia,Hallazgo en La Patagonia: un cráneo de 66 mill...,"Wed, 24 Jun 2026 23:59:38 +0000",https://www.pagina12.com.ar/2026/06/24/hallazg...
1,Página 12 - Ciencia,Científicos argentinos abren la posibilidad de...,"Sun, 16 Nov 2025 19:53:11 +0000",https://www.pagina12.com.ar/2025/11/16/cientif...
2,Página 12 - Ciencia,Alarma por el retroceso del glaciar Perito Moreno,"Thu, 07 Aug 2025 23:41:22 +0000",https://www.pagina12.com.ar/848097-alarma-por-...
3,Página 12 - Ciencia,"La investigadora del CONICET Sandra Díaz, entr...","Thu, 17 Apr 2025 00:52:24 +0000",https://www.pagina12.com.ar/818840-la-investig...
4,Página 12 - Ciencia,El futuro de la IA y su impacto en el conocimi...,"Sat, 19 Oct 2024 03:01:00 +0000",https://www.pagina12.com.ar/775639-el-futuro-d...
5,Página 12 - Universidad,Relatos sobre la crisis: vivir en tiempos cong...,"Sat, 23 Dec 2023 12:57:34 +0000",https://www.pagina12.com.ar/698097-relatos-sob...
6,Página 12 - Universidad,Estudian en la UNNE el uso alternativo antidia...,"Fri, 22 Dec 2023 10:01:12 +0000",https://www.pagina12.com.ar/697664-estudian-en...
7,Página 12 - Universidad,Una investigación científica advierte sobre pr...,"Fri, 22 Dec 2023 10:01:12 +0000",https://www.pagina12.com.ar/697660-una-investi...
8,Página 12 - Universidad,Breves,"Fri, 22 Dec 2023 10:00:15 +0000",https://www.pagina12.com.ar/697666-breves
9,Página 12 - Universidad,Proponen convertir residuos cítricos en una al...,"Fri, 22 Dec 2023 10:00:15 +0000",https://www.pagina12.com.ar/697662-proponen-co...


In [20]:
for source_name in sources:
    print("=" * 80)
    print(source_name.upper())
    print("=" * 80)

    display(
        df_feed_exploration[
            df_feed_exploration["source"] == source_name
        ][["title", "published"]]
    )

PÁGINA 12 - CIENCIA


,title,published
0,Hallazgo en La Patagonia: un cráneo de 66 mill...,"Wed, 24 Jun 2026 23:59:38 +0000"
1,Científicos argentinos abren la posibilidad de...,"Sun, 16 Nov 2025 19:53:11 +0000"
2,Alarma por el retroceso del glaciar Perito Moreno,"Thu, 07 Aug 2025 23:41:22 +0000"
3,"La investigadora del CONICET Sandra Díaz, entr...","Thu, 17 Apr 2025 00:52:24 +0000"
4,El futuro de la IA y su impacto en el conocimi...,"Sat, 19 Oct 2024 03:01:00 +0000"


PÁGINA 12 - UNIVERSIDAD


,title,published
5,Relatos sobre la crisis: vivir en tiempos cong...,"Sat, 23 Dec 2023 12:57:34 +0000"
6,Estudian en la UNNE el uso alternativo antidia...,"Fri, 22 Dec 2023 10:01:12 +0000"
7,Una investigación científica advierte sobre pr...,"Fri, 22 Dec 2023 10:01:12 +0000"
8,Breves,"Fri, 22 Dec 2023 10:00:15 +0000"
9,Proponen convertir residuos cítricos en una al...,"Fri, 22 Dec 2023 10:00:15 +0000"


AGENCIA UNQ - CAT 15


,title,published
10,Las máquinas ya baten récords de atletas en la...,"Tue, 25 Aug 2026 19:11:56 +0000"
11,"Cruzada del sur, el videojuego que recrea la c...","Tue, 25 Aug 2026 15:38:33 +0000"
12,"Qué esconde el proyecto “Edward Warchocki”, el...","Tue, 18 Aug 2026 16:51:37 +0000"
13,La Torre Eiffel derrumbada: Google Earth elimi...,"Mon, 10 Aug 2026 18:35:13 +0000"
14,El desarrollo global de reactores modulares es...,"Mon, 27 Jul 2026 02:16:25 +0000"


... (2 outputs left out)

AGENCIA UNQ - CAT 482


,title,published
20,¿Cree que su jefe es frío e insensible? Espere...,"Fri, 28 Mar 2025 16:15:06 +0000"
21,Científicos argentinos desarrollan una vincha ...,"Mon, 01 Jul 2024 20:33:18 +0000"
22,Una solución para millones: una startup rosari...,"Fri, 12 Apr 2024 14:03:11 +0000"
23,La empresa MSD y Unicef se unen para reducir l...,"Tue, 19 Mar 2024 17:24:25 +0000"
24,La pyme argentina Biocientífica lanzó un kit p...,"Tue, 26 Dec 2023 12:40:00 +0000"


UTN - REVISTA TECNOLOGÍA Y CIENCIA


,title,published
25,Modelado y resolución numérica de propiedades ...,"Wed, 15 Jul 2026 00:00:00 +0000"
26,M-SCORE: Un Modelo Cuantitativo y Explicable d...,"Mon, 20 Jul 2026 00:00:00 +0000"
27,Inteligencia artificial aplicada a la operació...,"Thu, 20 Aug 2026 00:00:00 +0000"
28,Arquitectura Híbrida Interpretable basada en W...,"Wed, 10 Jun 2026 00:00:00 +0000"
29,Uso de residuos de Moringa oleífera como adsor...,"Tue, 25 Aug 2026 00:00:00 +0000"


In [26]:
from bs4 import BeautifulSoup
from urllib.parse import urljoin


def discover_feeds(page_url):
    response = requests.get(
        page_url,
        timeout=15,
        headers={"User-Agent": "Mozilla/5.0"}
    )

    soup = BeautifulSoup(response.text, "html.parser")

    feeds = []

    for link in soup.find_all("link", rel="alternate"):
        content_type = link.get("type", "").lower()

        if "rss" in content_type or "atom" in content_type:
            feeds.append({
                "page_url": page_url,
                "feed_url": urljoin(page_url, link.get("href")),
                "type": content_type,
                "title": link.get("title")
            })

    return feeds

In [27]:
candidate_pages = {
    "Infobae - Tecno":
        "https://www.infobae.com/tecno/",

    "Clarín - Tecnología":
        "https://www.clarin.com/tecnologia/",

    "La Nación - Ciencia":
        "https://www.lanacion.com.ar/ciencia/",

    "La Nación - Tecnología":
        "https://www.lanacion.com.ar/tecnologia/",

    "Argentina Investiga":
        "https://argentinainvestiga.edu.ar/",
}

In [28]:
discovered = []

for name, url in candidate_pages.items():
    try:
        feeds = discover_feeds(url)

        if feeds:
            for feed in feeds:
                discovered.append({
                    "source": name,
                    **feed
                })
        else:
            discovered.append({
                "source": name,
                "page_url": url,
                "feed_url": None,
                "type": None,
                "title": None
            })

    except Exception as error:
        discovered.append({
            "source": name,
            "page_url": url,
            "feed_url": None,
            "type": None,
            "title": None,
            "error": str(error)
        })


df_discovered = pd.DataFrame(discovered)

df_discovered

,source,page_url,feed_url,type,title
0,Infobae - Tecno,https://www.infobae.com/tecno/,https://www.infobae.com/arc/outboundfeeds/rss/...,application/rss+xml,RSS de Tecno - Infobae
1,Clarín - Tecnología,https://www.clarin.com/tecnologia/,NaN,NaN,NaN
2,La Nación - Ciencia,https://www.lanacion.com.ar/ciencia/,NaN,NaN,NaN
3,La Nación - Tecnología,https://www.lanacion.com.ar/tecnologia/,NaN,NaN,NaN
4,Argentina Investiga,https://argentinainvestiga.edu.ar/,NaN,NaN,NaN


In [29]:
infobae_feed = df_discovered.loc[
    df_discovered["source"] == "Infobae - Tecno",
    "feed_url"
].iloc[0]

print(infobae_feed)

https://www.infobae.com/arc/outboundfeeds/rss/category/tecno/


In [30]:
validate_rss(
    "Infobae - Tecno",
    infobae_feed
)

{'source': 'Infobae - Tecno',
 'url': 'https://www.infobae.com/arc/outboundfeeds/rss/category/tecno/',
 'http_status': 200,
 'content_type': 'application/xml; charset=utf-8',
 'feed_title': 'Infobae.com',
 'entries': 100,
 'bozo': False,
 'valid': True}

In [31]:
show_feed(
    infobae_feed,
    limit=10
)

,title,published,link
0,Epic Games Store regala un nuevo juego de terr...,"Thu, 27 Aug 2026 16:28:47 +0000",https://www.infobae.com/tecno/2026/08/27/epic-...
1,Bill Gates advierte sobre la IA: “será el mayo...,"Thu, 27 Aug 2026 16:11:37 +0000",https://www.infobae.com/tecno/2026/08/27/bill-...
2,WhatsApp refuerza la seguridad de las cuentas ...,"Thu, 27 Aug 2026 16:10:29 +0000",https://www.infobae.com/tecno/2026/08/27/whats...
3,Gratis en Steam: el juego que combina superviv...,"Thu, 27 Aug 2026 15:41:10 +0000",https://www.infobae.com/tecno/2026/08/27/grati...
4,Truco para ampliar la cobertura del internet e...,"Thu, 27 Aug 2026 15:39:22 +0000",https://www.infobae.com/tecno/2026/08/27/truco...
5,Time reveló las 100 personalidades más influye...,"Thu, 27 Aug 2026 15:37:49 +0000",https://www.infobae.com/tecno/2026/08/27/time-...
6,Un informe reveló cómo 700 agentes de IA de Op...,"Thu, 27 Aug 2026 15:30:29 +0000",https://www.infobae.com/tecno/2026/08/27/un-in...
7,Qué presentaría Apple en su evento del 9 de se...,"Thu, 27 Aug 2026 15:29:16 +0000",https://www.infobae.com/tecno/2026/08/27/que-p...
8,A qué hora se estrena el avance extendido de G...,"Thu, 27 Aug 2026 15:03:11 +0000",https://www.infobae.com/tecno/2026/08/27/a-que...
9,La temperatura ideal del aire acondicionado en...,"Thu, 27 Aug 2026 14:49:07 +0000",https://www.infobae.com/tecno/2026/08/27/la-te...


In [32]:
candidate_feeds = {
    # Clarín
    "Clarín - Tecnología 1":
        "https://www.clarin.com/rss/tecnologia/",
    "Clarín - Tecnología 2":
        "https://www.clarin.com/rss/tecnologia.xml",

    # La Nación
    "La Nación - Tecnología 1":
        "https://www.lanacion.com.ar/arc/outboundfeeds/rss/category/tecnologia/",
    "La Nación - Ciencia 1":
        "https://www.lanacion.com.ar/arc/outboundfeeds/rss/category/ciencia/",

    # Argentina Investiga / WordPress-style candidates
    "Argentina Investiga - RSS":
        "https://argentinainvestiga.edu.ar/feed/",
    "Argentina Investiga - RSS2":
        "https://argentinainvestiga.edu.ar/?feed=rss2",
}

In [33]:
candidate_results = [
    validate_rss(name, url)
    for name, url in candidate_feeds.items()
]

df_candidates = pd.DataFrame(candidate_results)

df_candidates[
    [
        "source",
        "http_status",
        "content_type",
        "feed_title",
        "entries",
        "bozo",
        "valid"
    ]
]

,source,http_status,content_type,feed_title,entries,bozo,valid
0,Clarín - Tecnología 1,200,text/xml; charset=utf-8,Clarin.com - Home - Tecnología,10,False,True
1,Clarín - Tecnología 2,404,application/json; charset=utf-8,NaN,0,1,False
2,La Nación - Tecnología 1,200,application/xml; charset=utf-8,LA NACION,0,False,False
3,La Nación - Ciencia 1,200,application/xml; charset=utf-8,LA NACION,0,False,False
4,Argentina Investiga - RSS,404,text/html; charset=iso-8859-1,NaN,0,1,False
5,Argentina Investiga - RSS2,200,text/html; charset=ISO-8859-1,NaN,0,1,False


In [34]:
valid_candidates = df_candidates[
    df_candidates["valid"] == True
]

valid_candidates

,source,url,http_status,content_type,feed_title,entries,bozo,valid
0,Clarín - Tecnología 1,https://www.clarin.com/rss/tecnologia/,200,text/xml; charset=utf-8,Clarin.com - Home - Tecnología,10,False,True


In [36]:
show_feed(
    candidate_feeds["Clarín - Tecnología 1"],
    limit=10
)

,title,published,link
0,Tres modelos de iPhone dominan el ranking de c...,"Thu, 27 Aug 2026 08:46:05 +0000",https://www.clarin.com/tecnologia/modelos-ipho...
1,Meta bloqueará Instagram y Facebook para adole...,"Wed, 26 Aug 2026 15:13:57 +0000",https://www.clarin.com/tecnologia/meta-bloquea...
2,¿Puede explotar la batería de un celular? Qué ...,"Wed, 26 Aug 2026 08:45:51 +0000",https://www.clarin.com/tecnologia/puede-explot...
3,Cómo liberar espacio en el celular sin borrar ...,"Tue, 25 Aug 2026 08:45:34 +0000",https://www.clarin.com/tecnologia/liberar-espa...
4,"Medias de compresión: una ""segunda piel"" que a...","Mon, 24 Aug 2026 08:45:20 +0000",https://www.clarin.com/tecnologia/medias-compr...
5,Sonos Play: probamos el parlante portátil que ...,"Sun, 23 Aug 2026 09:01:06 +0000",https://www.clarin.com/tecnologia/sonos-play-p...
6,Un hacker publicó los primeros clips inéditos ...,"Sat, 22 Aug 2026 09:00:53 +0000",https://www.clarin.com/tecnologia/hacker-publi...
7,Filtran todos los detalles del iPhone 18 Pro a...,"Fri, 21 Aug 2026 08:46:35 +0000",https://www.clarin.com/tecnologia/filtran-deta...
8,“Palantir es como una caja negra”: la adverten...,"Wed, 19 Aug 2026 08:46:04 +0000",https://www.clarin.com/tecnologia/palantir-caj...
9,"Quién es Peter Thiel, el magnate tecnológico y...","Tue, 18 Aug 2026 13:37:15 +0000",https://www.clarin.com/tecnologia/peter-thiel-...


In [37]:
url = candidate_feeds["La Nación - Tecnología 1"]

response = requests.get(
    url,
    timeout=15,
    headers={"User-Agent": "Mozilla/5.0"}
)

print(response.text[:3000])

<?xml version="1.0" encoding="UTF-8"?><rss xmlns:atom="http://www.w3.org/2005/Atom" xmlns:content="http://purl.org/rss/1.0/modules/content/" xmlns:dc="http://purl.org/dc/elements/1.1/" xmlns:sy="http://purl.org/rss/1.0/modules/syndication/" version="2.0" xmlns:media="http://search.yahoo.com/mrss/"><channel><title><![CDATA[LA NACION]]></title><link>https://www.lanacion.com.ar</link><atom:link href="https://www.lanacion.com.ar/arc/outboundfeeds/rss/category/tecnologia/" rel="self" type="application/rss+xml"/><description><![CDATA[LA NACION News Feed]]></description><lastBuildDate>Thu, 27 Aug 2026 16:43:50 +0000</lastBuildDate><language>es</language><ttl>1</ttl><sy:updatePeriod>hourly</sy:updatePeriod><sy:updateFrequency>1</sy:updateFrequency></channel></rss>


In [38]:
selected_sources = [
    {
        "name": "Infobae - Tecno",
        "category": "technology",
        "feed_url": "https://www.infobae.com/arc/outboundfeeds/rss/category/tecno/",
        "website_url": "https://www.infobae.com/tecno/",
        "status": "active",
    },
    {
        "name": "Clarín - Tecnología",
        "category": "technology",
        "feed_url": "https://www.clarin.com/rss/tecnologia/",
        "website_url": "https://www.clarin.com/tecnologia/",
        "status": "active",
    },
    {
        "name": "Agencia UNQ - Tecnología",
        "category": "technology",
        "feed_url": "https://agencia.unq.edu.ar/?feed=rss2&cat=15",
        "website_url": "https://agencia.unq.edu.ar/?cat=15",
        "status": "active",
    },
    {
        "name": "Agencia UNQ - Ciencias",
        "category": "science",
        "feed_url": "https://agencia.unq.edu.ar/?feed=rss2&cat=8",
        "website_url": "https://agencia.unq.edu.ar/?cat=8",
        "status": "active",
    },
    {
        "name": "Agencia UNQ - Empresas",
        "category": "business",
        "feed_url": "https://agencia.unq.edu.ar/?feed=rss2&cat=482",
        "website_url": "https://agencia.unq.edu.ar/?cat=482",
        "status": "low_activity",
    },
    {
        "name": "UTN - Revista Tecnología y Ciencia",
        "category": "science",
        "feed_url": "https://rtyc.utn.edu.ar/index.php/rtyc/gateway/plugin/WebFeedGatewayPlugin/rss2",
        "website_url": "https://rtyc.utn.edu.ar/index.php/rtyc",
        "status": "active",
    },
    {
        "name": "Página 12 - Ciencia",
        "category": "science",
        "feed_url": "https://www.pagina12.com.ar/arc/outboundfeeds/rss/secciones/ciencia/notas",
        "website_url": "https://www.pagina12.com.ar/ciencia/",
        "status": "low_activity",
    },
]

df_selected_sources = pd.DataFrame(selected_sources)

df_selected_sources

,name,category,feed_url,website_url,status
0,Infobae - Tecno,technology,https://www.infobae.com/arc/outboundfeeds/rss/...,https://www.infobae.com/tecno/,active
1,Clarín - Tecnología,technology,https://www.clarin.com/rss/tecnologia/,https://www.clarin.com/tecnologia/,active
2,Agencia UNQ - Tecnología,technology,https://agencia.unq.edu.ar/?feed=rss2&cat=15,https://agencia.unq.edu.ar/?cat=15,active
3,Agencia UNQ - Ciencias,science,https://agencia.unq.edu.ar/?feed=rss2&cat=8,https://agencia.unq.edu.ar/?cat=8,active
4,Agencia UNQ - Empresas,business,https://agencia.unq.edu.ar/?feed=rss2&cat=482,https://agencia.unq.edu.ar/?cat=482,low_activity
5,UTN - Revista Tecnología y Ciencia,science,https://rtyc.utn.edu.ar/index.php/rtyc/gateway...,https://rtyc.utn.edu.ar/index.php/rtyc,active
6,Página 12 - Ciencia,science,https://www.pagina12.com.ar/arc/outboundfeeds/...,https://www.pagina12.com.ar/ciencia/,low_activity


In [39]:
discarded_sources = [
    {
        "name": "Página 12 - Universidad",
        "reason": "RSS válido, pero sin publicaciones recientes."
    },
    {
        "name": "La Nación - Tecnología",
        "reason": "RSS válido, pero actualmente devuelve 0 entradas."
    },
    {
        "name": "La Nación - Ciencia",
        "reason": "RSS válido, pero actualmente devuelve 0 entradas."
    },
    {
        "name": "Argentina Investiga",
        "reason": "No se encontró un RSS funcional."
    },
]

pd.DataFrame(discarded_sources)

,name,reason
0,Página 12 - Universidad,"RSS válido, pero sin publicaciones recientes."
1,La Nación - Tecnología,"RSS válido, pero actualmente devuelve 0 entradas."
2,La Nación - Ciencia,"RSS válido, pero actualmente devuelve 0 entradas."
3,Argentina Investiga,No se encontró un RSS funcional.


------------------

In [ ]:
#CELDA 6
try:
    feeds_df = spark.sql("""
    SELECT feed_url
    FROM dbo.rss_links
    WHERE is_active = true
    """)

    feeds = [r["feed_url"] for r in feeds_df.collect()]
    print("Feeds activos:", feeds)

    schema = StructType([
        StructField("feed_url", StringType(), False),
        StructField("ingestion_ts", TimestampType(), False),
        StructField("http_status", IntegerType(), True),
        StructField("raw_xml", StringType(), True),
        StructField("is_success", BooleanType(), False),
        StructField("error_message", StringType(), True),
        StructField("content_hash", StringType(), True),
    ])

    for feed_url in feeds:
        print(f"\nProcesando feed: {feed_url}")

        ingestion_ts = datetime.now(timezone.utc)

        http_status = None
        raw_xml = None
        is_success = True
        error_message = None
        content_hash = None

        try:
            resp = requests.get(feed_url, timeout=30, headers={"User-Agent": "FabricRSSIngest/1.0"})
            http_status = resp.status_code
            raw_xml = resp.text

            if http_status >= 400:
                is_success = False
                error_message = f"HTTP error {http_status}"

        except Exception as e:
            is_success = False
            error_message = str(e)

        if raw_xml is not None:
            content_hash = hashlib.sha256(raw_xml.encode("utf-8")).hexdigest()

        last_hash = None
        try:
            feed_url_sql = feed_url.replace("'", "''")
            last_row = spark.sql(f"""
                SELECT content_hash
                FROM dbo.bronze_rss
                WHERE feed_url = '{feed_url_sql}'
                ORDER BY ingestion_ts DESC
                LIMIT 1
            """).collect()

            if len(last_row) > 0:
                last_hash = last_row[0]["content_hash"]
        except Exception as e:
            print("No se pudo leer el último hash:", str(e))

        if content_hash is not None and content_hash == last_hash:
            print("No hubo cambios. No se inserta.")
            continue

        print("Contenido nuevo (o primera vez / hash previo NULL). Insertando...")

        df = spark.createDataFrame([Row(
            feed_url=feed_url,
            ingestion_ts=ingestion_ts,
            http_status=http_status,
            raw_xml=raw_xml,
            is_success=is_success,
            error_message=error_message,
            content_hash=content_hash
        )], schema=schema)

        df.write.format("delta").mode("append").saveAsTable("essoc_lk_bronze.dbo.bronze_rss")
        rows_written += 1

    end_time = datetime.now()

    success_log(
    pipeline_name,
    notebook_name,
    layer,
    source_name,
    execution_id,
    start_time,
    rows_written
    )   

except Exception as e:

    error_log(
        pipeline_name,
        notebook_name,
        layer,
        source_name,
        execution_id,
        start_time,
        rows_written,
        e
    )

    raise

In [ ]:
def prepare_keywords_for_matching(keywords: list[str]) -> list[dict]:
    patterns = []

    for kw in keywords:
        raw = (kw or "").strip()
        if not raw:
            continue

        low = raw.lower()

        if low.startswith("@") or low.startswith("#"):
            rx = re.compile(
                rf"(?<!\w){re.escape(low)}(?!\w)",
                flags=re.IGNORECASE
            )
            patterns.append({
                "raw": raw,
                "type": "tag",
                "rx": rx
            })
            continue

        rx = re.compile(
            rf"\b{re.escape(low)}\b",
            flags=re.IGNORECASE
        )
        patterns.append({
            "raw": raw,
            "type": "phrase",
            "rx": rx
        })

    return patterns

try:

    # 1) Traer feeds
    feeds = spark.sql("""
    SELECT source_name, feed_url
    FROM essoc_lk_bronze.dbo.rss_links
    WHERE is_active = true
    """).collect()


    # ====== PARAMETERS ======

    KEYWORDS_TABLE = "essoc_lk_bronze.dbo.keywords"
    DETRACTORES_TABLE = "essoc_lk_bronze.dbo.detractores"


    def get_text(elem, tag):
        node = elem.find(tag)
        if node is None or node.text is None:
            return None
        return node.text.strip()


    all_rows = []


    # 2) Para cada feed, tomar el último XML y extraer items
    for r in feeds:
        source_name = r["source_name"]
        feed_url = r["feed_url"]

        last = spark.sql(f"""
        SELECT ingestion_ts, raw_xml
        FROM essoc_lk_bronze.dbo.bronze_rss
        WHERE feed_url = '{feed_url.replace("'", "''")}'
        ORDER BY ingestion_ts DESC
        LIMIT 1
        """).collect()

        if not last:
            print(f"[WARN] No hay datos en bronze para {source_name} ({feed_url})")
            continue

        ingestion_ts = last[0]["ingestion_ts"]
        raw_xml = last[0]["raw_xml"]

        try:
            root = ET.fromstring(raw_xml)
        except Exception as e:
            print(f"[ERROR] XML inválido para {source_name}: {e}")
            continue

        channel = root.find("channel")
        items = channel.findall("item") if channel is not None else []

        for it in items:
            all_rows.append({
                "source_name": source_name,
                "feed_url": feed_url,
                "ingestion_ts": ingestion_ts,
                "title": get_text(it, "title"),
                "link": get_text(it, "link"),
                "description": get_text(it, "description"),
                "guid": get_text(it, "guid"),
                "pub_date_raw": get_text(it, "pubDate"),
            })


    items_df = spark.createDataFrame(all_rows)


    items_df = (
        items_df
        .withColumn("guid", F.trim(F.col("guid")))
        .withColumn("link", F.trim(F.col("link")))
        .withColumn("title", F.trim(F.col("title")))

        .withColumn(
            "canonical_link",
            F.lower(
                F.regexp_replace(F.col("link"), "/$", "")
            )
        )

        .withColumn(
            "article_id",
            F.sha2(
                F.coalesce(
                    F.col("canonical_link"),
                    F.concat_ws("||", F.col("feed_url"), F.col("guid")),
                    F.concat_ws("||", F.col("feed_url"), F.col("title"), F.col("pub_date_raw"))
                ),
                256
            )
        )

        .withColumn("published_at", F.to_timestamp("pub_date_raw"))

        .drop("pub_date_raw")
    )

    # ====================================================
    # DATA QUALITY CHECKS - ITEMS
    # ====================================================

    total_items = items_df.count()

    invalid_article_id_null = items_df.filter(F.col("article_id").isNull()).count()
    invalid_article_id_empty = items_df.filter(F.trim(F.col("article_id")) == "").count()
    invalid_published_at_null = items_df.filter(F.col("published_at").isNull()).count()
    invalid_title_desc_both_null = items_df.filter(
        F.col("title").isNull() & F.col("description").isNull()
    ).count()
    invalid_title_desc_both_empty = items_df.filter(
        (F.coalesce(F.trim(F.col("title")), F.lit("")) == "") &
        (F.coalesce(F.trim(F.col("description")), F.lit("")) == "")
    ).count()
    invalid_title_desc_too_short = items_df.filter(
        (F.length(F.coalesce(F.trim(F.col("title")), F.lit(""))) < 2) &
        (F.length(F.coalesce(F.trim(F.col("description")), F.lit(""))) < 2)
    ).count()

    print("====== DATA QUALITY - RSS ITEMS ======")
    print("Total registros:", total_items)
    print("article_id null:", invalid_article_id_null)
    print("article_id vacío:", invalid_article_id_empty)
    print("published_at null:", invalid_published_at_null)
    print("title y description null:", invalid_title_desc_both_null)
    print("title y description vacíos:", invalid_title_desc_both_empty)
    print("title y description < 2 chars:", invalid_title_desc_too_short)

    items_df_valid = (
        items_df
        .filter(F.col("article_id").isNotNull())
        .filter(F.trim(F.col("article_id")) != "")
        .filter(F.col("published_at").isNotNull())
        .filter(
            ~(
                (F.coalesce(F.trim(F.col("title")), F.lit("")) == "") &
                (F.coalesce(F.trim(F.col("description")), F.lit("")) == "")
            )
        )
        .filter(
            ~(
                (F.length(F.coalesce(F.trim(F.col("title")), F.lit(""))) < 2) &
                (F.length(F.coalesce(F.trim(F.col("description")), F.lit(""))) < 2)
            )
        )
    )

    valid_items = items_df_valid.count()
    print("items válidos:", valid_items)
    print("items descartados:", total_items - valid_items)


    table_name = "essoc_lk_silver.dbo.silver_rss_stagging"

    if spark.catalog.tableExists(table_name):

        delta_table = DeltaTable.forName(spark, table_name)

        (
            delta_table.alias("t")
            .merge(
                items_df_valid.alias("s"),
                "t.article_id = s.article_id"
            )
            .whenNotMatchedInsertAll()
            .execute()
        )

        print("MERGE ejecutado correctamente")

    else:
        print("Creando dbo.silver_rss_stagging (primera vez)")

        (
            items_df_valid.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(table_name)
        )


    spark.sql("""
    SELECT
      source_name,
      COUNT(*) AS rows,
      COUNT(DISTINCT article_id) AS distinct_articles
    FROM essoc_lk_silver.dbo.silver_rss_stagging
    GROUP BY source_name
    ORDER BY source_name
    """).show()


    spark.sql("""
    SELECT article_id, COUNT(*)
    FROM essoc_lk_silver.dbo.silver_rss_stagging
    GROUP BY article_id
    HAVING COUNT(*) > 1
    """).show()


    df_kw = (
        spark.table(KEYWORDS_TABLE)
        .select("keyword")
        .withColumn("keyword", F.trim(F.col("keyword")))
        .filter(F.col("keyword").isNotNull() & (F.col("keyword") != ""))
        .dropDuplicates(["keyword"])
    )

    df_detractors = (
        spark.table(DETRACTORES_TABLE)
        .filter(
            (F.col("is_active") == True) &
            (F.col("detractor").isNotNull()) &
            (F.trim(F.col("detractor")) != "")
        )
        .select(
            F.lower(
                F.regexp_replace(F.trim(F.col("detractor")), r"^@", "")
            ).alias("detractor_username")
        )
        .dropDuplicates()
    )

    df_kw.show()


    keywords = [r["keyword"] for r in df_kw.select("keyword").collect()]


    def _normalize(text: str) -> str:
        if not text:
            return ""
        text = text.lower().strip()
        text = unicodedata.normalize("NFKD", text)
        return "".join(c for c in text if not unicodedata.combining(c))


    def extract_matched_keywords(tweet_text: str, kw_patterns: list[dict], max_matches: int = 50) -> list[str]:

        txt = _normalize(tweet_text)
        if not txt:
            return []

        matches = []
        for p in kw_patterns:
            if p["rx"].search(txt):
                matches.append(p["raw"])
                if len(matches) >= max_matches:
                    break

        return matches


    KW_PATTERNS = prepare_keywords_for_matching(keywords)


    df_articles = spark.table("essoc_lk_silver.dbo.silver_rss_stagging")

    extract_keywords_udf = F.udf(
        lambda x: extract_matched_keywords(x, KW_PATTERNS),
        ArrayType(StringType())
    )


    df_articles = df_articles.withColumn(
        "keywords",
        extract_keywords_udf(F.col("description"))
    )


    df_keywords_exploded = (
        df_articles
        .select(
            "article_id",
            "source_name",
            "feed_url",
            "title",
            "link",
            "description",
            "published_at",
            "ingestion_ts",
            explode_outer(F.col("keywords")).alias("keyword")
        )
        .filter(F.col("keyword").isNotNull())
    )

    # ====================================================
    # DATA QUALITY CHECKS - RSS KEYWORDS EXPLODED
    # ====================================================

    total_keywords = df_keywords_exploded.count()

    invalid_kw_article_id_null = df_keywords_exploded.filter(F.col("article_id").isNull()).count()
    invalid_kw_article_id_empty = df_keywords_exploded.filter(F.trim(F.col("article_id")) == "").count()
    invalid_kw_keyword_null = df_keywords_exploded.filter(F.col("keyword").isNull()).count()
    invalid_kw_keyword_empty = df_keywords_exploded.filter(F.trim(F.col("keyword")) == "").count()

    print("====== DATA QUALITY - RSS KEYWORDS ======")
    print("Total registros:", total_keywords)
    print("article_id null:", invalid_kw_article_id_null)
    print("article_id vacío:", invalid_kw_article_id_empty)
    print("keyword null:", invalid_kw_keyword_null)
    print("keyword vacío:", invalid_kw_keyword_empty)

    df_keywords_exploded_valid = (
        df_keywords_exploded
        .filter(F.col("article_id").isNotNull())
        .filter(F.trim(F.col("article_id")) != "")
        .filter(F.col("keyword").isNotNull())
        .filter(F.trim(F.col("keyword")) != "")
    )

    valid_keywords = df_keywords_exploded_valid.count()
    print("keywords válidos:", valid_keywords)
    print("keywords descartados:", total_keywords - valid_keywords)


    table_keywords = "essoc_lk_silver.dbo.silver_rss"

    if spark.catalog.tableExists(table_keywords):
        delta_kw = DeltaTable.forName(spark, table_keywords)

        (
            delta_kw.alias("t")
            .merge(
                df_keywords_exploded_valid.alias("s"),
                "t.article_id = s.article_id AND t.keyword <=> s.keyword"
            )
            .whenNotMatchedInsertAll()
            .execute()
        )

        print("MERGE ejecutado en silver_rss")

    else:
        print("Creando silver_rss (primera vez)")

        (
            df_keywords_exploded_valid.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(table_keywords)
        )


    df_unique = (
        spark.table("essoc_lk_silver.dbo.silver_rss")
        .dropDuplicates(["article_id"])
    )


    table_unique = "essoc_lk_silver.dbo.silver_rss_unique"

    if spark.catalog.tableExists(table_unique):
        delta_unique = DeltaTable.forName(spark, table_unique)

        (
            delta_unique.alias("t")
            .merge(
                df_unique.alias("s"),
                "t.article_id = s.article_id"
            )
            .whenNotMatchedInsertAll()
            .execute()
        )
        print("MERGE ejecutado en silver_rss_unique")

    else:
        print("Creando silver_rss_unique (primera vez)")
        (
            df_unique.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(table_unique)
        )


    rows_written = spark.table("essoc_lk_silver.dbo.silver_rss_unique").count()


    success_log(
        pipeline_name,
        notebook_name,
        layer,
        source_name,
        execution_id,
        start_time,
        rows_written
    )


except Exception as e:

    error_log(
        pipeline_name,
        notebook_name,
        layer,
        source_name,
        execution_id,
        start_time,
        rows_written,
        e
    )

    raise


# Tomar primera fila por article_id (elimina duplicados del explode)
df_unique = (
    spark.table("essoc_lk_silver.dbo.silver_rss")
    .dropDuplicates(["article_id"])
)

table_unique = "essoc_lk_silver.dbo.silver_rss_unique"

if spark.catalog.tableExists(table_unique):
    delta_unique = DeltaTable.forName(spark, table_unique)

    (
        delta_unique.alias("t")
        .merge(
            df_unique.alias("s"),
            "t.article_id = s.article_id"
        )
        .whenNotMatchedInsertAll()
        .execute()
    )
    print("MERGE ejecutado en silver_rss_unique")

else:
    print("Creando silver_rss_unique (primera vez)")
    (
        df_unique.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(table_unique)
    )